# Síntese da Evolução AAD: notebooks 1 a 25

Este notebook foi criado para **contar a história completa do projeto**, de forma técnica e didática:

- o que entrou em cada notebook;
- o que foi descartado ou perdeu força;
- por que certas técnicas sobreviveram até a arquitetura final;
- como o projeto saiu de testes isolados e virou um **benchmark robusto, multi-fonte e orientado a Edge AI**.

A ideia aqui não é repetir todos os experimentos, e sim **explicar a lógica da evolução**.


## Como ler este notebook

Pense nesta síntese em 4 camadas:

1. **Linha do tempo**: o que cada notebook adicionou.
2. **Mapa de técnicas**: quando cada técnica entrou, virou núcleo ou perdeu força.
3. **Tabela de descarte/manutenção**: por que algumas ideias foram mantidas e outras não.
4. **Veredito final provável**: qual arquitetura ficou mais coerente para o artigo e para a banca.


In [ ]:
from pathlib import Path
import json
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (16, 8)
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 11

ROOT = Path("/Users/emanoelspanhol/Downloads/V2 benchmarking 2")
notebook_paths = sorted(
    [p for p in ROOT.glob("[0-9]* - *.ipynb")],
    key=lambda p: int(p.name.split(" - ")[0]),
)

inventory_rows = []
for p in notebook_paths:
    nb = json.loads(p.read_text(encoding="utf-8"))
    first_text = ""
    for cell in nb.get("cells", []):
        src = "".join(cell.get("source", [])).strip()
        if src:
            first_text = " ".join(src.split())[:160]
            break
    inventory_rows.append(
        {
            "id": int(p.name.split(" - ")[0]),
            "arquivo": p.name,
            "abertura": first_text,
        }
    )

inventory_df = pd.DataFrame(inventory_rows)
display(Markdown("## Inventário dos notebooks 1 a 25"))
display(inventory_df)


## 1. Linha do tempo curada da evolução

A tabela abaixo é a espinha dorsal desta síntese. Ela resume, notebook a notebook, a principal decisão metodológica tomada no projeto.


In [ ]:
evolution_rows = [
    {
        "id": 1,
        "fase": "Exploração inicial",
        "arquivo": "1 - Teste supervisionado",
        "marco": "Primeiro pipeline supervisionado",
        "entrou": "Leitura de áudio, janelas iniciais, classificação supervisionada",
        "saiu_ou_perdeu_forca": "Nada descartado ainda; fase de prova de conceito",
        "por_que": "Verificar se existia separabilidade básica entre normal e anômalo",
        "legado": "Mostrou que o problema era viável",
    },
    {
        "id": 2,
        "fase": "Exploração inicial",
        "arquivo": "2 - Teste_Não__Supervisionado",
        "marco": "Primeiro eixo não supervisionado",
        "entrou": "Fatiamento e análise sem rótulos de falha no treino",
        "saiu_ou_perdeu_forca": "A ideia de depender sempre de rótulos completos",
        "por_que": "Aproximar o problema do cenário real/first-shot",
        "legado": "Abriu o trilho do DCASE-like",
    },
    {
        "id": 3,
        "fase": "Exploração inicial",
        "arquivo": "3 - Cenário_2_Baseline",
        "marco": "Baseline controlado com dados sintéticos",
        "entrou": "Benchmarking inicial em cenário Cloud",
        "saiu_ou_perdeu_forca": "Confiança excessiva em dados artificiais",
        "por_que": "Criar referência mínima para comportamento do pipeline",
        "legado": "Serviu como solo de comparação",
    },
    {
        "id": 4,
        "fase": "Exploração inicial",
        "arquivo": "4 - Cenário_2_V1_Analise_Espectograma_De_Mel",
        "marco": "Entrada do espectrograma Mel",
        "entrou": "Mel/log-Mel como base tempo-frequência",
        "saiu_ou_perdeu_forca": "Representações muito brutas do sinal",
        "por_que": "Mel organiza o áudio de forma compacta, robusta e compatível com MFCC/NMF/XAI",
        "legado": "Virou base estável da arquitetura final",
    },
    {
        "id": 5,
        "fase": "Exploração inicial",
        "arquivo": "5 - Cenário_2_V2_Analise_CWT",
        "marco": "Teste com wavelets/CWT",
        "entrou": "Análise multirresolução para transientes",
        "saiu_ou_perdeu_forca": "A ideia de que CWT seria obrigatória",
        "por_que": "Investigar eventos curtos e não estacionários",
        "legado": "Permaneceu como referência exploratória, não como eixo final",
    },
    {
        "id": 6,
        "fase": "Exploração inicial",
        "arquivo": "6 - Cenário_2_V3_Analise_STFT",
        "marco": "Teste explícito com STFT",
        "entrou": "Leitura espectral clássica",
        "saiu_ou_perdeu_forca": "A STFT como protagonista única",
        "por_que": "Comparar bases espectrais antes de consolidar a representação",
        "legado": "Ajudou a justificar por que Mel virou a base principal",
    },
    {
        "id": 7,
        "fase": "Arquitetura híbrida",
        "arquivo": "7 - Cenário_2_V1_Benchmarking_Mel_MFCC_CWT_Hibrido",
        "marco": "Primeira fusão híbrida",
        "entrou": "Mel + MFCC + CWT em benchmark estruturado",
        "saiu_ou_perdeu_forca": "Pipelines isolados por técnica",
        "por_que": "Misturar representações complementares do sinal",
        "legado": "Nasceu a ideia de Super-Vector multimodal",
    },
    {
        "id": 8,
        "fase": "Arquitetura híbrida",
        "arquivo": "8 - Cenário_2_V2_Benchmarking_Mel_MFCC_CWT_Hibrido_NMF",
        "marco": "Entrada do NMF",
        "entrou": "Erro de reconstrução NMF como canal estrutural",
        "saiu_ou_perdeu_forca": "Dependência de features apenas espectrais",
        "por_que": "Medir o quanto um som quebra a normalidade aprendida",
        "legado": "Virou um dos canais mais elegantes do vetor final",
    },
    {
        "id": 9,
        "fase": "Arquitetura híbrida",
        "arquivo": "9 - Cenário_2_V3_Benchmarking_Mel_MFCC_CWT_Hibrido_NMF_Rolamentos",
        "marco": "Entrada do Kaggle de rolamentos",
        "entrou": "Kaggle real como base pública principal",
        "saiu_ou_perdeu_forca": "Validação só em cenário controlado",
        "por_que": "Levar o benchmark para dados públicos reais",
        "legado": "O Kaggle virou a espinha dorsal supervisionada",
    },
    {
        "id": 10,
        "fase": "Arquitetura híbrida",
        "arquivo": "10 - Cenário_2_V4_Benchmarking_Mel_MFCC_CWT_Hibrido_NMF_Validação_Final",
        "marco": "Validação com teste final próprio",
        "entrou": "Teste cego com seus arquivos coletados",
        "saiu_ou_perdeu_forca": "Conforto de medir só treino/validação interna",
        "por_que": "Forçar a arquitetura a encarar mudança de domínio",
        "legado": "Começou a separar laboratório de realidade",
    },
    {
        "id": 11,
        "fase": "Frentes de endurecimento",
        "arquivo": "11 - Frente_1_(Regularização_GAMMA)",
        "marco": "Regularização + lógica probabilística",
        "entrou": "Regularização L2 e discussão de Gamma",
        "saiu_ou_perdeu_forca": "Fronteiras frágeis e limiares excessivamente empíricos",
        "por_que": "Tornar o classificador mais estável e o limiar mais justificável",
        "legado": "Preparou o terreno para o threshold industrial",
    },
    {
        "id": 12,
        "fase": "Frentes de endurecimento",
        "arquivo": "12 - Frente_2_(CNN_+_GMM)",
        "marco": "Teste de CNN e entrada do GMM",
        "entrou": "CNN como desafiante e GMM no trilho não supervisionado",
        "saiu_ou_perdeu_forca": "A crença de que só ML clássico bastaria para tudo",
        "por_que": "Comparar modelagem densa com estatística multimodal",
        "legado": "O GMM sobreviveu; a CNN como motor final não",
    },
    {
        "id": 13,
        "fase": "Frentes de endurecimento",
        "arquivo": "13 - Frente_3_(HHT_Limpeza_Isolamento)",
        "marco": "Entrada do HHT + UKF",
        "entrou": "Purificação acústica não linear/estocástica",
        "saiu_ou_perdeu_forca": "Pré-processamento raso demais",
        "por_que": "Elevar SNR e destacar transientes mecânicos reais",
        "legado": "Virou peça central da arquitetura campeã",
    },
    {
        "id": 14,
        "fase": "Frentes de endurecimento",
        "arquivo": "14 - Frente_4_(Data_Augmentation_via_Mixup)",
        "marco": "Entrada do Mixup",
        "entrou": "Data augmentation para robustez de fronteira",
        "saiu_ou_perdeu_forca": "Treino excessivamente rígido ao domínio original",
        "por_que": "Diminuir pânico do modelo diante de domain shift",
        "legado": "Permaneceu relevante, sobretudo no supervisionado",
    },
    {
        "id": 15,
        "fase": "Frentes de endurecimento",
        "arquivo": "15 - Frente_5_(pAuc,_Score_DCASE)",
        "marco": "Mudança da régua de avaliação",
        "entrou": "pAUC@0.1 e score DCASE",
        "saiu_ou_perdeu_forca": "O hábito de julgar tudo só por F1/AUC global",
        "por_que": "A indústria tolera mal falsos positivos; o DCASE também força esse olhar",
        "legado": "Métrica industrial virou critério de decisão",
    },
    {
        "id": 16,
        "fase": "Frentes de endurecimento",
        "arquivo": "16 - Frente_6_Threshold_Gamma_+_FPR_alvo",
        "marco": "Threshold estatístico formal",
        "entrou": "Distribuição Gamma + FPR alvo",
        "saiu_ou_perdeu_forca": "Limiar fixo e heurístico",
        "por_que": "Reduzir fadiga de alarmes com base matemática",
        "legado": "Virou parte essencial do ramo não supervisionado",
    },
    {
        "id": 17,
        "fase": "Frentes de endurecimento",
        "arquivo": "17 - Frente_7_Outlier_Exposure_Real_Acumulativo_via_DCASE",
        "marco": "OE real com DCASE",
        "entrou": "Outlier Exposure e alinhamento mais forte com DCASE",
        "saiu_ou_perdeu_forca": "Validação fechada demais no universo Kaggle/Drive",
        "por_que": "Endurecer fronteiras e ampliar repertório de ruídos/ambientes",
        "legado": "Ficou como estratégia útil, mas não obrigatória no final",
    },
    {
        "id": 18,
        "fase": "Frentes de endurecimento",
        "arquivo": "18 - Frente_8_Baselines_Não_Supervisionados",
        "marco": "Ampliação dos baselines não supervisionados",
        "entrou": "Mahalanobis, GMM, Isolation Forest e comparativos mais sérios",
        "saiu_ou_perdeu_forca": "A noção de um único detector universal",
        "por_que": "Entender onde cada hipótese estatística funciona ou falha",
        "legado": "Preparou o veredito posterior entre Mahalanobis e GMM",
    },
    {
        "id": 19,
        "fase": "Frentes de endurecimento",
        "arquivo": "19 - Frente_9_pretrain_e_augmentation_Copia",
        "marco": "Pré-treino e augmentation mais agressivos",
        "entrou": "Pré-treino, augmentation ampliado e abertura para extratores profundos",
        "saiu_ou_perdeu_forca": "A ideia de que bastava aumentar dados sem refinar representação",
        "por_que": "Buscar generalização mais profunda",
        "legado": "Abriu espaço para Tiny-AST como módulo auxiliar",
    },
    {
        "id": 20,
        "fase": "Frentes de endurecimento",
        "arquivo": "20 - Frente_10_(Ajustes_do_xai)_Copia",
        "marco": "Consolidação do XAI",
        "entrou": "Heatmaps, hotspots H1/H2/H3 e leitura física das bandas",
        "saiu_ou_perdeu_forca": "Modelos caixa-preta sem explicação operacional",
        "por_que": "Traduzir score em diagnóstico compreensível para manutenção",
        "legado": "Virou argumento-chave do artigo e da arquitetura",
    },
    {
        "id": 21,
        "fase": "Consolidação final",
        "arquivo": "21 - Frente_11_FINAL_v_2_Copia",
        "marco": "Grande consolidação multicritério",
        "entrou": "Benchmark final robusto, governança experimental e comparação ampla",
        "saiu_ou_perdeu_forca": "Testes soltos e conclusões prematuras",
        "por_que": "Encontrar a combinação mais forte antes da limpeza arquitetural",
        "legado": "Foi a arena de batalha de onde saiu a arquitetura campeã",
    },
    {
        "id": 22,
        "fase": "Consolidação final",
        "arquivo": "22 - Arquitetura_Campea_AAD_DCASE_MIMII",
        "marco": "Arquitetura campeã limpa e didática",
        "entrou": "Pipeline focado em HHT+UKF + RPCA + Super-Vector + NMF",
        "saiu_ou_perdeu_forca": "Excesso de modelos pesados como decisão final",
        "por_que": "Transformar a arena experimental em produto reprodutível",
        "legado": "Nasceu a versão enxuta e explicável do sistema",
    },
    {
        "id": 23,
        "fase": "Consolidação final",
        "arquivo": "23 - Cópia_de_Arquitetura_Campea_AAD_DCASE_MIMII",
        "marco": "Refino e cópia operacional da campeã",
        "entrou": "Ajustes de redação, ingestão e organização didática",
        "saiu_ou_perdeu_forca": "Ruído estrutural do notebook anterior",
        "por_que": "Dar mais clareza à operação e ao relatório",
        "legado": "Serviu como ponte para o veredito final",
    },
    {
        "id": 24,
        "fase": "Consolidação final",
        "arquivo": "24 - Veredito_Final_AAD_Protocolos",
        "marco": "Separação formal em dois regimes",
        "entrou": "Protocolo supervisionado multi-fonte e não supervisionado DCASE-like",
        "saiu_ou_perdeu_forca": "A ideia de um campeão universal único",
        "por_que": "A evidência mostrou que o melhor modelo depende do regime",
        "legado": "Fixou XGBoost como campeão supervisionado e abriu espaço para GMM no não supervisionado",
    },
    {
        "id": 25,
        "fase": "Consolidação final",
        "arquivo": "25 - Veredito_Final_AAD_Protocolos_V2_1",
        "marco": "Benchmark multi-fonte fechado com MIMII",
        "entrou": "MIMII com -6 dB, 0 dB e 6 dB; reforço do teste cego multi-fonte",
        "saiu_ou_perdeu_forca": "Dependência de poucas fontes públicas",
        "por_que": "Endurecer o benchmark e fechar a história experimental",
        "legado": "Virou a base mais madura para o artigo final",
    },
]

evolution_df = pd.DataFrame(evolution_rows)
display(evolution_df)


In [ ]:
phase_order = ["Exploração inicial", "Arquitetura híbrida", "Frentes de endurecimento", "Consolidação final"]
phase_colors = {
    "Exploração inicial": "#90caf9",
    "Arquitetura híbrida": "#80cbc4",
    "Frentes de endurecimento": "#ffcc80",
    "Consolidação final": "#ef9a9a",
}

fig, ax = plt.subplots(figsize=(18, 4))
for _, row in evolution_df.iterrows():
    ax.scatter(
        row["id"],
        phase_order.index(row["fase"]) + 1,
        s=500,
        color=phase_colors[row["fase"]],
        edgecolor="black",
        linewidth=1.2,
    )
    ax.text(
        row["id"],
        phase_order.index(row["fase"]) + 1.18,
        str(row["id"]),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_yticks(range(1, len(phase_order) + 1))
ax.set_yticklabels(phase_order)
ax.set_xticks(evolution_df["id"])
ax.set_xlabel("Notebook")
ax.set_title("Linha do Tempo do Projeto: do experimento inicial ao veredito final")
ax.grid(True, axis="x", alpha=0.2)
sns.despine(left=False, bottom=False)
plt.show()


## 2. Mapa de técnicas ao longo dos 25 notebooks

O gráfico abaixo mostra **quando cada técnica apareceu** e qual papel ela assumiu:

- `0` = ausente
- `1` = teste exploratório
- `2` = uso ativo importante
- `3` = núcleo da arquitetura final
- `4` = módulo auxiliar / apoio


In [ ]:
notebooks = list(range(1, 26))
matrix = {
    "Janelas deslizantes": [1,1,1,1,1,1,2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3],
    "Mel / log-Mel":        [0,0,0,3,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],
    "MFCC":                 [0,0,0,1,1,1,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],
    "CWT / wavelets":       [0,0,0,0,3,1,2,2,2,2,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0],
    "STFT explícita":       [0,0,0,0,0,3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0],
    "NMF":                  [0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],
    "RPCA":                 [0,0,0,0,0,0,0,1,1,1,1,1,2,2,2,2,2,2,2,2,3,3,3,3,3],
    "HHT + UKF":            [0,0,0,0,0,0,0,0,0,0,0,1,3,3,3,3,3,3,3,3,3,3,3,3,3],
    "Mixup":                [0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3,3],
    "Gamma + FPR alvo":     [0,0,0,0,0,0,0,0,0,0,2,1,1,1,2,3,3,3,3,3,3,3,3,3,3],
    "pAUC / Score DCASE":   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3,3,3,3,3,3],
    "XGBoost":              [2,0,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,3,3,3],
    "Mahalanobis":          [0,2,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,3,2,2,3,3,3,2,2],
    "GMM":                  [0,0,0,0,0,0,0,0,0,0,0,2,1,1,1,1,1,2,2,2,2,2,2,3,3],
    "Isolation Forest":     [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,2,2,2,2,2],
    "OE / DCASE externo":   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,2,2,2,2,2,2,2,2],
    "Tiny-AST":             [0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,2,2,2,4,4,4,4],
    "XAI":                  [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,3,3,3,3,3],
    "Kaggle":               [0,0,0,1,1,1,1,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],
    "DCASE":                [0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,3,3,3,3,3,3,3,3,3],
    "MIMII":                [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3],
}

heat_df = pd.DataFrame(matrix, index=notebooks).T
cmap = sns.color_palette(["#f5f5f5", "#bbdefb", "#90caf9", "#26a69a", "#ffcc80"], as_cmap=True)

plt.figure(figsize=(22, 12))
sns.heatmap(heat_df, cmap=cmap, linewidths=0.5, linecolor="white", cbar_kws={"label": "estado da técnica"})
plt.title("Mapa de técnicas ao longo dos 25 notebooks")
plt.xlabel("Notebook")
plt.ylabel("Técnica")
plt.show()


In [ ]:
goal_rows = [
    ("Mel / log-Mel", "representação compacta e estável"),
    ("MFCC", "capturar timbre e dinâmica"),
    ("CWT / wavelets", "detectar transientes não estacionários"),
    ("STFT explícita", "comparar bases espectrais"),
    ("NMF", "medir quebra estrutural da normalidade"),
    ("RPCA", "separar fundo estável de evento raro"),
    ("HHT + UKF", "limpar ruído estocástico e realçar impactos"),
    ("Mixup", "aumentar robustez ao domain shift"),
    ("Gamma + FPR alvo", "controlar falsos positivos com base matemática"),
    ("pAUC / Score DCASE", "avaliar zona de baixo falso alarme"),
    ("Mahalanobis", "modelar normalidade unimodal"),
    ("GMM", "modelar normalidade multimodal"),
    ("Tiny-AST", "injetar semântica profunda como apoio"),
    ("XAI", "traduzir score em fenômeno físico"),
]

goal_df = pd.DataFrame(goal_rows, columns=["técnica", "motivo_principal"])
display(Markdown("## Motivo principal de entrada de cada técnica"))
display(goal_df)

counts = goal_df["motivo_principal"].value_counts().reset_index()
counts.columns = ["motivo_principal", "quantidade"]
plt.figure(figsize=(16, 8))
sns.barplot(data=counts, y="motivo_principal", x="quantidade", color="#42a5f5")
plt.title("O projeto evoluiu puxado por quais necessidades?")
plt.xlabel("Quantidade de técnicas associadas")
plt.ylabel("Necessidade de engenharia")
plt.show()


## 3. O que foi descartado, reduzido ou reposicionado

Aqui está a parte mais importante para o artigo: não basta dizer o que entrou. Precisamos explicar **por que certas escolhas perderam centralidade**.


In [ ]:
discard_rows = [
    {
        "elemento": "Dados sintéticos como eixo principal",
        "veredito": "descartado como protocolo principal",
        "motivo": "Servem para baseline, mas não representam o ruído e a variabilidade do chão de fábrica.",
        "o_que_ficou": "Apenas como referência inicial de comportamento.",
    },
    {
        "elemento": "CWT como base central",
        "veredito": "reduzido a técnica exploratória",
        "motivo": "Ajudou a estudar transientes, mas adicionou custo/complexidade sem se consolidar como base mais estável que Mel.",
        "o_que_ficou": "Valor analítico e comparativo.",
    },
    {
        "elemento": "STFT explícita como protagonista",
        "veredito": "reduzida",
        "motivo": "Mel/log-Mel entregou melhor compactação e integração com MFCC, NMF, RPCA e XAI.",
        "o_que_ficou": "Referência espectral clássica.",
    },
    {
        "elemento": "CNN / deep learning pesado como decisor final",
        "veredito": "descartado como núcleo final",
        "motivo": "Maior custo computacional, mais risco de memorização de ruído e pior encaixe com Edge/TinyML.",
        "o_que_ficou": "Aprendizado profundo reaproveitado como módulo auxiliar, não como motor central.",
    },
    {
        "elemento": "Mahalanobis como campeão universal",
        "veredito": "reposicionado",
        "motivo": "Funciona bem com normalidade homogênea, mas sofre quando a normalidade fica multimodal no teste cego multi-fonte.",
        "o_que_ficou": "Baseline valioso e referência conceitual para normalidade unimodal.",
    },
    {
        "elemento": "Limiar empírico fixo",
        "veredito": "descartado",
        "motivo": "Não controla o custo real do falso alarme na indústria.",
        "o_que_ficou": "Foi substituído por Gamma + FPR alvo.",
    },
    {
        "elemento": "Outlier Exposure como etapa obrigatória",
        "veredito": "tornado opcional",
        "motivo": "Ajuda em alguns cenários, mas não se mostrou condição obrigatória para a versão final do produto.",
        "o_que_ficou": "Ferramenta de robustez, sob governança experimental.",
    },
    {
        "elemento": "Tiny-AST como classificador final",
        "veredito": "descartado como campeão final",
        "motivo": "O custo não compensou como motor principal; porém os embeddings e a saliência visual agregaram muito valor.",
        "o_que_ficou": "Módulo auxiliar de embeddings + XAI.",
    },
]

discard_df = pd.DataFrame(discard_rows)
display(discard_df)


In [ ]:
final_roles = pd.DataFrame(
    [
        ("núcleo final", 8),
        ("baseline importante", 3),
        ("módulo auxiliar", 2),
        ("exploratório / descartado", 5),
    ],
    columns=["papel", "quantidade"],
)

plt.figure(figsize=(9, 9))
colors = ["#26a69a", "#42a5f5", "#ffca28", "#ef5350"]
plt.pie(
    final_roles["quantidade"],
    labels=final_roles["papel"],
    autopct="%1.0f%%",
    startangle=90,
    colors=colors,
    textprops={"fontsize": 12},
)
plt.title("Distribuição final das técnicas no projeto")
plt.show()


## 4. Arquitetura final provável

A síntese dos 25 notebooks indica que o projeto terminou em uma **arquitetura híbrida com dois regimes de operação**:

- **Regime supervisionado industrial**: campeão `XGBoost`
- **Regime não supervisionado DCASE-like / multi-domínio**: campeão `GMM + Gamma`

A base comum é a mesma:

`Áudio bruto -> HHT + UKF -> janelas deslizantes -> Mel/log-Mel -> RPCA + MFCC + NMF + módulo auxiliar Tiny-AST -> Super-Vector -> decisão`


In [ ]:
final_arch_df = pd.DataFrame(
    [
        ("1. Entrada", "Áudio bruto multi-fonte", "Receber sinal real de Kaggle, DCASE, MIMII e coleta própria", "comum aos dois regimes"),
        ("2. Purificação", "HHT + UKF", "Limpar ruído estocástico e destacar impactos curtos", "núcleo"),
        ("3. Segmentação", "Janelas deslizantes", "Capturar eventos locais e preservar resolução temporal", "núcleo"),
        ("4. Base espectral", "Mel / log-Mel", "Representação compacta e consolidada para áudio", "núcleo"),
        ("5. Separação", "RPCA", "Separar fundo estável de eventos raros", "núcleo"),
        ("6. Vetor acústico", "MFCC + bandas Mel + estatísticas + NMF", "Destilar o áudio em um Super-Vector leve e interpretável", "núcleo"),
        ("7. Semântica auxiliar", "Tiny-AST embeddings + saliência", "Injetar semântica profunda sem tornar o pipeline dependente de um transformer", "auxiliar"),
        ("8A. Decisão supervisionada", "XGBoost", "Campeão quando há rótulos e falhas conhecidas", "campeão supervisionado"),
        ("8B. Decisão não supervisionada", "GMM + Gamma", "Campeão quando a normalidade é multimodal e o treino é só com normal", "campeão não supervisionado"),
        ("8C. Baseline não supervisionado", "Mahalanobis + Gamma", "Referência importante para normalidade homogênea/unimodal", "baseline importante"),
        ("9. Explicabilidade", "XAI com hotspots, bandas e saliência", "Traduzir score em diagnóstico físico", "núcleo"),
    ],
    columns=["etapa", "componente", "função", "status"],
)

display(final_arch_df)


In [ ]:
fig, ax = plt.subplots(figsize=(22, 5))
boxes = [
    ("Áudio bruto", 0.05, "#90caf9"),
    ("HHT + UKF", 0.18, "#80cbc4"),
    ("Janelas", 0.31, "#80cbc4"),
    ("Mel/log-Mel", 0.44, "#ffe082"),
    ("RPCA + MFCC + NMF", 0.59, "#ffcc80"),
    ("Super-Vector", 0.75, "#ffab91"),
    ("XGBoost / GMM", 0.90, "#ce93d8"),
]

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

for label, x, color in boxes:
    ax.text(
        x,
        0.55,
        label,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.6", fc=color, ec="black", lw=1.5),
        fontsize=13,
        fontweight="bold",
    )

for i in range(len(boxes) - 1):
    x1 = boxes[i][1] + 0.06
    x2 = boxes[i + 1][1] - 0.07
    ax.annotate("", xy=(x2, 0.55), xytext=(x1, 0.55), arrowprops=dict(arrowstyle="->", lw=2))

ax.set_title("Fluxo final provável da arquitetura", fontsize=20, pad=18)
plt.show()


## 5. Os dois protocolos finais que fazem sentido


In [ ]:
protocol_df = pd.DataFrame(
    [
        (
            "Benchmark supervisionado multi-fonte",
            "Kaggle + DCASE + MIMII (com split por arquivo, máquina e fonte)",
            "XGBoost",
            "F1, AUC, pAUC@0.1, matriz de confusão, resultado por fonte, leave-one-source-out",
            "Quando há anomalias rotuladas e interesse operacional direto",
        ),
        (
            "Benchmark não supervisionado alinhado ao DCASE",
            "Treino só com normal; anomalias só em validação/teste",
            "GMM + Gamma",
            "pAUC@0.1, AUC, F1, análise por origem, teste cego multi-fonte",
            "Quando o foco é first-shot / noise-aware unsupervised ASD",
        ),
        (
            "Teste cego operacional",
            "Drive cego + fatias reservadas de Kaggle, DCASE e MIMII",
            "Comparar XGBoost, GMM, Mahalanobis e baselines",
            "F1, AUC, pAUC@0.1 e leitura por fonte",
            "Quando o objetivo é medir robustez real e domain shift",
        ),
    ],
    columns=["protocolo", "dados", "campeão esperado", "o que reportar", "uso principal"],
)

display(protocol_df)


In [ ]:
tradeoff_df = pd.DataFrame(
    [
        ("XGBoost", 2, 4, 4, "campeão supervisionado"),
        ("Mahalanobis", 1, 2, 4, "baseline importante"),
        ("GMM", 2, 4, 3, "campeão não supervisionado"),
        ("Isolation Forest", 2, 2, 3, "baseline secundário"),
        ("Tiny-AST end-to-end", 5, 3, 1, "não recomendado como motor final"),
    ],
    columns=["modelo", "custo_edge", "robustez_multi_dominio", "explicabilidade", "papel"],
)

palette = {
    "campeão supervisionado": "#26a69a",
    "baseline importante": "#42a5f5",
    "campeão não supervisionado": "#ef5350",
    "baseline secundário": "#ffca28",
    "não recomendado como motor final": "#ab47bc",
}

plt.figure(figsize=(14, 10))
for _, row in tradeoff_df.iterrows():
    plt.scatter(
        row["custo_edge"],
        row["robustez_multi_dominio"],
        s=500,
        color=palette[row["papel"]],
        edgecolors="black",
        linewidth=1.2,
    )
    plt.text(
        row["custo_edge"] + 0.05,
        row["robustez_multi_dominio"] + 0.05,
        row["modelo"],
        fontsize=12,
    )

plt.xlim(0.5, 5.5)
plt.ylim(0.5, 5.5)
plt.xticks([1, 2, 3, 4, 5])
plt.yticks([1, 2, 3, 4, 5])
plt.xlabel("Custo computacional na borda (escala qualitativa)")
plt.ylabel("Robustez sob multi-domínio (escala qualitativa)")
plt.title("Mapa qualitativo de trade-off entre os candidatos finais")
plt.show()


## 6. Contribuições finais do trabalho

Se você quiser fechar a narrativa com força, esta é a tabela mais importante: ela transforma a trajetória experimental em **contribuições científicas e de engenharia**.


In [ ]:
contribution_df = pd.DataFrame(
    [
        (
            "Governança experimental anti-vazamento",
            "Separação por arquivo, máquina e fonte; teste cego multi-fonte",
            "Evita overfitting ilusório e aproxima a avaliação do chão de fábrica",
            "10, 17, 21, 24, 25",
        ),
        (
            "Purificação acústica robusta",
            "HHT + UKF antes da análise local",
            "Melhora o contraste entre falha e ruído estocástico",
            "13, 21, 22, 24, 25",
        ),
        (
            "Representação híbrida leve e explicável",
            "Mel + MFCC + RPCA + NMF no Super-Vector",
            "Entrega alta densidade informacional sem depender de redes profundas pesadas",
            "7, 8, 13, 21, 22, 24, 25",
        ),
        (
            "Avaliação industrial orientada a baixo falso alarme",
            "pAUC@0.1 + Gamma + FPR alvo",
            "Foca no que realmente importa para adoção em fábrica",
            "15, 16, 21, 24, 25",
        ),
        (
            "Dupla via de decisão",
            "XGBoost no supervisionado e GMM + Gamma no não supervisionado",
            "Mostra que o melhor detector depende do regime de operação",
            "18, 21, 24, 25",
        ),
        (
            "Explicabilidade acionável",
            "XAI com hotspots, bandas e saliência",
            "Traduz score em fenômeno físico compreensível para manutenção",
            "20, 21, 22, 24, 25",
        ),
        (
            "Uso pragmático de deep learning",
            "Tiny-AST como embeddings + XAI auxiliar, não como motor final obrigatório",
            "Aproveita semântica profunda preservando custo/explicabilidade",
            "19, 22, 24, 25",
        ),
    ],
    columns=["contribuição", "materialização no pipeline", "ganho prático", "notebooks-evidência"],
)

display(contribution_df)


In [ ]:
contrib_plot = contribution_df.copy()
contrib_plot["força_argumentativa"] = [5, 5, 5, 5, 4, 5, 4]

plt.figure(figsize=(15, 8))
sns.barplot(
    data=contrib_plot.sort_values("força_argumentativa", ascending=False),
    x="força_argumentativa",
    y="contribuição",
    palette="viridis",
)
plt.title("Força argumentativa das contribuições para o artigo/banca")
plt.xlabel("Importância qualitativa na defesa")
plt.ylabel("Contribuição")
plt.xlim(0, 5.2)
plt.show()


## 7. Mapa de evidências: de qual notebook sai cada argumento?

Esta seção é útil para duas coisas:

- **escrever o artigo sem se perder**;
- **garantir reprodutibilidade narrativa**, isto é, conseguir apontar exatamente onde cada conclusão foi construída.


In [ ]:
evidence_df = pd.DataFrame(
    [
        ("Viabilidade inicial supervisionada", "1", "prova de conceito"),
        ("Viabilidade inicial não supervisionada", "2", "abertura do regime first-shot"),
        ("Baseline controlado", "3", "referência de solo"),
        ("Escolha do Mel como base principal", "4, 6", "comparação entre representações"),
        ("Valor exploratório da CWT", "5", "transientes e não estacionariedade"),
        ("Nascimento do híbrido", "7", "fusão de Mel + MFCC + CWT"),
        ("Entrada do NMF", "8", "canal estrutural"),
        ("Entrada do Kaggle real", "9", "benchmark público real"),
        ("Teste final próprio / domain shift inicial", "10", "saída do laboratório puro"),
        ("L2 / endurecimento do supervisionado", "11", "regularização"),
        ("Entrada do GMM", "12", "multimodalidade da normalidade"),
        ("Entrada do HHT + UKF", "13", "purificação acústica"),
        ("Entrada do Mixup", "14", "robustez a domínio"),
        ("Mudança da régua para pAUC", "15", "critério industrial"),
        ("Threshold Gamma com FPR alvo", "16", "controle matemático do falso alarme"),
        ("Alinhamento com DCASE / OE", "17", "validação externa mais dura"),
        ("Comparação séria entre detectores não supervisionados", "18", "Mahalanobis vs GMM vs outros"),
        ("Pré-treino e abertura ao Tiny-AST", "19", "semântica auxiliar"),
        ("Consolidação do XAI", "20", "diagnóstico visual/físico"),
        ("Arena final multicritério", "21", "benchmark consolidado"),
        ("Arquitetura campeã limpa", "22 e 23", "pipeline reprodutível"),
        ("Veredito por protocolo", "24 e 25", "decisão final por regime"),
    ],
    columns=["argumento", "notebook-fonte", "papel"],
)

display(evidence_df)


## 8. Reprodutibilidade: o que precisa estar congelado

Para o seu fechamento ficar acadêmico e defensável, não basta ter bons gráficos. É importante mostrar **como outra pessoa repetiria a lógica**.


In [ ]:
repro_df = pd.DataFrame(
    [
        ("Fontes de dados", "Kaggle, DCASE, MIMII e Drive cego definidos explicitamente", "sem troca silenciosa de base"),
        ("Unidade de separação", "split por arquivo, máquina e fonte", "evita data leakage entre janelas irmãs"),
        ("Etapa global", "HHT + UKF aplicados antes da análise local", "condicionamento fixo do sinal"),
        ("Etapa local", "janelas deslizantes com duração e hop definidos", "mesma granularidade temporal"),
        ("Representação-base", "Mel/log-Mel", "garante comparabilidade entre notebooks finais"),
        ("Extração por janela", "RPCA, MFCC, NMF e, quando aplicável, Tiny-AST auxiliar", "mesmo vetor de entrada"),
        ("Regime supervisionado", "XGBoost + regularização + Mixup", "comparabilidade do campeão supervisionado"),
        ("Regime não supervisionado", "GMM + Gamma; Mahalanobis como baseline", "comparabilidade do campeão DCASE-like"),
        ("Métricas", "F1, AUC, pAUC@0.1 e leitura por fonte", "comparação justa entre cenários"),
        ("Teste cego", "multi-fonte com subconjuntos reservados", "evidência real de generalização"),
    ],
    columns=["bloco", "o que congelar", "por que importa"],
)

display(repro_df)


In [ ]:
run_order_df = pd.DataFrame(
    [
        (1, "Rodar o notebook 25", "Fechar benchmark multi-fonte com Kaggle, DCASE, MIMII e Drive cego"),
        (2, "Extrair tabelas finais", "Salvar métricas globais, por fonte e teste cego"),
        (3, "Executar ablação RPCA", "Comparar com vs sem RPCA"),
        (4, "Executar ablação Tiny-AST auxiliar", "Comparar com vs sem embeddings auxiliares"),
        (5, "Preencher esta síntese", "Trazer os números finais para o notebook 26"),
        (6, "Escrever artigo", "Usar notebook 26 como narrativa e notebook 25 como evidência numérica"),
    ],
    columns=["ordem", "ação", "objetivo"],
)

display(run_order_df)

plt.figure(figsize=(14, 6))
sns.lineplot(data=run_order_df, x="ordem", y=[1] * len(run_order_df), marker="o", linewidth=3, color="#26a69a")
for _, row in run_order_df.iterrows():
    plt.text(row["ordem"], 1.03, row["ação"], ha="center", va="bottom", fontsize=11, rotation=20)
plt.yticks([])
plt.xlabel("Sequência recomendada para fechamento reprodutível")
plt.title("Ordem prática para fechar o pipeline, os resultados e o artigo")
plt.ylim(0.94, 1.08)
plt.show()


## 9. Decisão operacional: o que usar no final final

Se o objetivo agora é **fechar o projeto com consistência**, a decisão mais segura é esta:


In [ ]:
use_df = pd.DataFrame(
    [
        ("Base arquitetural", "HHT + UKF + janelas deslizantes + Mel/log-Mel + RPCA + MFCC + NMF", "é o núcleo mais consistente ao longo da evolução"),
        ("Campeão supervisionado", "XGBoost", "melhor equilíbrio entre desempenho, leveza e explicabilidade"),
        ("Campeão não supervisionado", "GMM + Gamma", "mais adequado quando a normalidade fica multimodal"),
        ("Baseline obrigatório", "Mahalanobis + Gamma", "precisa aparecer para mostrar o limite da hipótese unimodal"),
        ("Deep learning", "Tiny-AST apenas como embeddings + XAI auxiliar", "agrega semântica sem sequestrar o pipeline"),
        ("Métrica principal industrial", "pAUC@0.1", "melhor alinhamento com custo de falso alarme"),
        ("Teste de realidade", "teste cego multi-fonte por origem", "é onde se prova robustez real"),
    ],
    columns=["camada", "decisão recomendada", "justificativa"],
)

display(use_df)


## 10. O que dizer no artigo e na banca

A síntese dos notebooks 1 a 25 permite sustentar quatro mensagens fortes:

1. **O projeto evoluiu de provas de conceito isoladas para um benchmark multi-fonte e anti-vazamento.**
2. **A arquitetura final não é um modelo mágico único, mas uma solução híbrida com dois regimes de operação.**
3. **A robustez veio principalmente do DSP, da representação vetorial e da governança experimental, não do aumento cego de complexidade.**
4. **O deep learning ajudou melhor como módulo auxiliar semântico e visual do que como motor final obrigatório.**


### Arguição final executiva

> O percurso experimental demonstrou que o ganho central do projeto não surgiu da simples troca de modelos, mas do amadurecimento progressivo do pipeline como sistema. Partimos de testes supervisionados e não supervisionados isolados, com foco inicial em viabilidade, e evoluímos para um benchmark multi-fonte, com blindagem contra vazamento, teste cego e avaliação orientada por pAUC@0.1. Ao longo desse processo, o projeto descartou a dependência de representações e modelos mais pesados como eixo obrigatório, priorizando uma arquitetura híbrida em que o sinal é primeiro purificado, depois segmentado e só então convertido em atributos leves, explicáveis e robustos. Esse caminho consolidou HHT+UKF, janelas deslizantes, Mel/log-Mel, RPCA, MFCC e NMF como o núcleo de representação.

> A principal contribuição prática desta experiência foi mostrar que a escolha do melhor detector depende do regime de operação. Quando há rótulos e interesse operacional direto, o XGBoost oferece o melhor equilíbrio entre desempenho, custo e interpretabilidade. Quando o problema é não supervisionado e a normalidade se torna multimodal pela entrada de fontes como DCASE e MIMII, o GMM + Gamma mostra-se mais adequado do que a hipótese unimodal do Mahalanobis. Em paralelo, o Tiny-AST demonstrou valor real como módulo auxiliar, fornecendo embeddings e suporte ao XAI, mas sem justificar sua adoção como motor central.

> Assim, a arquitetura final recomendada não é um modelo único e universal, mas um arranjo reprodutível de dupla via, sustentado por governança experimental, DSP robusto, representação vetorial leve e avaliação industrialmente coerente. É isso que transforma a experiência acumulada nos 25 notebooks em uma contribuição acadêmica sólida e, ao mesmo tempo, em uma solução defensável para uso real em Edge AI.


### Texto-base para o artigo

**Narrativa curta:**

> A trajetória experimental do projeto partiu de testes supervisionados e não supervisionados simples, passou por análises comparativas de diferentes representações do sinal, consolidou uma arquitetura híbrida com Mel, MFCC, NMF e, posteriormente, incorporou frentes sucessivas de robustez, como HHT+UKF, Mixup, pAUC@0.1, limiar via Distribuição Gamma e XAI. Ao longo desse percurso, técnicas como CWT, STFT explícita e modelos profundos mais pesados foram importantes para exploração, mas perderam centralidade por custo computacional, risco de memorização de ruído e menor aderência ao objetivo de Edge Computing. O resultado final não foi um único campeão universal, mas uma arquitetura de dupla via: XGBoost como campeão supervisionado industrial e GMM + Gamma como campeão não supervisionado em cenário multi-domínio alinhado ao DCASE.

**Mensagem científica central:**

> Quando a normalidade acústica é relativamente homogênea, a Distância de Mahalanobis permanece competitiva. Entretanto, à medida que o benchmark foi endurecido com múltiplas fontes e maior variabilidade de domínio, a normalidade deixou de ser unimodal, tornando o GMM mais adequado para o regime não supervisionado. Em paralelo, o uso de embeddings do Tiny-AST mostrou que o deep learning agregou semântica útil, mas sua principal contribuição ocorreu como módulo auxiliar, e não como substituto do núcleo híbrido baseado em DSP e representação vetorial leve.


### Próximos passos antes do artigo final

- Rodar a ablação **com RPCA vs sem RPCA**.
- Rodar a ablação **com Tiny-AST auxiliar vs sem Tiny-AST auxiliar**.
- Congelar as tabelas finais dos dois protocolos:
  - supervisionado multi-fonte;
  - não supervisionado DCASE-like.
- Reportar o teste cego multi-fonte **por origem**:
  - Drive;
  - Kaggle;
  - DCASE;
  - MIMII.
- Usar este notebook como guia narrativo e os notebooks 24/25 como fonte dos resultados numéricos finais.
